# Stage 6 V1 — TabM vs сохранённый GBDT baseline

## Исследовательский вопрос

Даёт ли один современный TabM на **тех же 47 разрешённых признаках** материальное улучшение относительно сохранённого Stage 1 XGBoost baseline?

Это один controlled experiment. `Q_B1_norm` и `Q_B2_norm` запрещены, final test не материализуется и не используется. До запуска notebook нет result artifacts.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tabm
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, TensorDataset


def project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Не найден корень проекта с pyproject.toml.')


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def hash_indices(values: np.ndarray) -> str:
    return hashlib.sha256(np.asarray(values, dtype=np.int64).tobytes()).hexdigest()


def set_fold_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True)


ROOT = project_root()
DATASET = ROOT / 'data' / 'raw' / 'Data_final.xlsb'
BASELINE_PATH = ROOT / 'reports' / 'generated' / 'stage1_baseline_results_V2.json'
GENERATED_DIR = ROOT / 'reports' / 'generated'
SUMMARY_DIR = ROOT / 'reports' / 'summary'
EXPECTED_DATASET_SHA256 = 'fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930'
EXPECTED_WORKING_INDEX_SHA256 = '80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45'
TARGET = 'DefMark'
IDENTIFIER = 'INN'
FORBIDDEN_FEATURES = ['Q_B1_norm', 'Q_B2_norm']
OUTER_SEED = 42
FOLD_SEEDS = {1: 43, 2: 44, 3: 45}
MAX_EPOCHS = 1000
PATIENCE = 16
RUNTIME_BUDGET_SECONDS = 8 * 60 * 60
TABM_CONFIG = {
    'arch_type': 'tabm', 'k': 32, 'n_blocks': 3, 'd_block': 512,
    'activation': 'ReLU', 'dropout': 0.10, 'start_scaling_init': 'random-signs',
    'num_embeddings': None, 'd_out': 2, 'input_dtype': 'float32',
    'optimizer': 'AdamW', 'lr': 0.002, 'weight_decay': 0.0003,
    'betas': (0.9, 0.999), 'eps': 1e-8, 'gradient_clip_global_norm': 1.0,
    'batch_size': 256, 'share_training_batches': True, 'max_epochs': MAX_EPOCHS,
    'amp': False, 'torch_compile': False, 'scheduler': None, 'warmup': None,
    'class_weights': None, 'sampling': None,
}

with BASELINE_PATH.open(encoding='utf-8') as handle:
    baseline = json.load(handle)
FEATURES = baseline['допустимые_признаки']
if len(FEATURES) != 47 or any(name in FEATURES for name in FORBIDDEN_FEATURES):
    raise ValueError('Нарушен Stage 1 contract: требуется ровно 47 разрешённых признаков.')
BASELINE_XGB = baseline['модели']['XGBoost']
BASELINE_XGB_METRICS = BASELINE_XGB['итоговые_метрики']
BASELINE_XGB_FOLDS = BASELINE_XGB['метрики_фолдов']
print({'python': platform.python_version(), 'torch': torch.__version__, 'tabm': tabm.__version__, 'numpy': np.__version__})
print('Устройство:', 'cpu; AMP и torch.compile выключены')


{'python': '3.12.2', 'torch': '2.13.0+cpu', 'tabm': '0.0.3', 'numpy': '2.5.2'}
Устройство: cpu; AMP и torch.compile выключены


## Locked protocol

- Dataset and Stage 1 feature identity are checked before training; NaN/non-finite values stop the run.
- Outer CV: 3-fold `StratifiedKFold(shuffle=True, random_state=42)`; fold seeds 43/44/45.
- Epoch selection uses only a stratified 90/10 split inside each outer train fold. The outer validation is predicted once, after refit on the full outer train for `best_epoch + 1` epochs.
- The eight-hour CPU limit is hard: the run stops and records `runtime_budget_exceeded`.

In [2]:
class RuntimeBudgetExceeded(RuntimeError):
    pass


def check_budget(started_at: float) -> None:
    if time.perf_counter() - started_at > RUNTIME_BUDGET_SECONDS:
        raise RuntimeBudgetExceeded('runtime_budget_exceeded')


def make_model(n_features: int) -> tabm.TabM:
    return tabm.TabM.make(
        n_num_features=n_features,
        cat_cardinalities=None,
        arch_type=TABM_CONFIG['arch_type'],
        k=TABM_CONFIG['k'],
        n_blocks=TABM_CONFIG['n_blocks'],
        d_block=TABM_CONFIG['d_block'],
        activation=TABM_CONFIG['activation'],
        dropout=TABM_CONFIG['dropout'],
        start_scaling_init=TABM_CONFIG['start_scaling_init'],
        num_embeddings=None,
        d_out=TABM_CONFIG['d_out'],
    )


def shared_loader(x: np.ndarray, y: np.ndarray, seed: int, shuffle: bool) -> DataLoader:
    generator = torch.Generator(device='cpu').manual_seed(seed)
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(y.astype(np.int64)))
    return DataLoader(
        dataset, batch_size=TABM_CONFIG['batch_size'], shuffle=shuffle,
        generator=generator, num_workers=0, drop_last=False,
    )


def fit_one_epoch(model: tabm.TabM, loader: DataLoader, optimizer: torch.optim.Optimizer, started_at: float) -> None:
    model.train()
    for x_batch, y_batch in loader:
        check_budget(started_at)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x_batch.float())  # (batch, 32 heads, 2 classes)
        targets = y_batch[:, None].expand(-1, model.k).reshape(-1)
        loss = F.cross_entropy(logits.reshape(-1, 2), targets)  # loss for each head, then mean
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), TABM_CONFIG['gradient_clip_global_norm'])
        optimizer.step()


@torch.inference_mode()
def positive_probabilities(model: tabm.TabM, x: np.ndarray, started_at: float) -> np.ndarray:
    model.eval()
    tensor = torch.from_numpy(x).float()
    chunks: list[np.ndarray] = []
    for start in range(0, len(tensor), TABM_CONFIG['batch_size']):
        check_budget(started_at)
        logits = model(tensor[start : start + TABM_CONFIG['batch_size']])
        probabilities = torch.softmax(logits, dim=-1).mean(dim=1)[:, 1]
        chunks.append(probabilities.cpu().numpy())
        check_budget(started_at)
    check_budget(started_at)
    return np.concatenate(chunks)


def select_best_epoch(x_outer: np.ndarray, y_outer: np.ndarray, fold_seed: int, started_at: float) -> tuple[int, float]:
    fit_idx, inner_idx = train_test_split(
        np.arange(len(y_outer)), test_size=0.10, stratify=y_outer, random_state=fold_seed,
    )
    set_fold_seed(fold_seed)
    model = make_model(x_outer.shape[1])
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=TABM_CONFIG['lr'], weight_decay=TABM_CONFIG['weight_decay'],
        betas=TABM_CONFIG['betas'], eps=TABM_CONFIG['eps'],
    )
    loader = shared_loader(x_outer[fit_idx], y_outer[fit_idx], fold_seed, shuffle=True)
    best_epoch, best_auc, stale_epochs = 0, float('-inf'), 0
    for epoch in range(MAX_EPOCHS):
        fit_one_epoch(model, loader, optimizer, started_at)
        inner_probability = positive_probabilities(model, x_outer[inner_idx], started_at)
        auc = float(roc_auc_score(y_outer[inner_idx], inner_probability))
        if auc > best_auc:  # strict improvement; min_delta = 0
            best_epoch, best_auc, stale_epochs = epoch, auc, 0
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                break
    del model, optimizer
    return best_epoch, best_auc


def refit_and_predict(x_train: np.ndarray, y_train: np.ndarray, x_valid: np.ndarray, fold_seed: int, best_epoch: int, started_at: float) -> np.ndarray:
    set_fold_seed(fold_seed)  # reset before full outer-train refit
    model = make_model(x_train.shape[1])
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=TABM_CONFIG['lr'], weight_decay=TABM_CONFIG['weight_decay'],
        betas=TABM_CONFIG['betas'], eps=TABM_CONFIG['eps'],
    )
    loader = shared_loader(x_train, y_train, fold_seed, shuffle=True)
    for _ in range(best_epoch + 1):
        fit_one_epoch(model, loader, optimizer, started_at)
    prediction = positive_probabilities(model, x_valid, started_at)
    del model, optimizer
    return prediction


def metrics(y_true: np.ndarray, probability: np.ndarray) -> dict[str, float]:
    predicted = (probability >= 0.5).astype(np.int64)
    auc = float(roc_auc_score(y_true, probability))
    return {
        'ROC-AUC': auc, 'Gini': 2.0 * auc - 1.0,
        'PR-AUC': float(average_precision_score(y_true, probability)),
        'Precision@0.5': float(precision_score(y_true, predicted, zero_division=0)),
        'Recall@0.5': float(recall_score(y_true, predicted, zero_division=0)),
        'F1@0.5': float(f1_score(y_true, predicted, zero_division=0)),
    }


def decide(tabm_metrics: dict[str, float], folds: list[dict[str, object]], runtime_seconds: float) -> str:
    delta_gini = tabm_metrics['Gini'] - float(BASELINE_XGB_METRICS['Gini'])
    delta_pr_auc = tabm_metrics['PR-AUC'] - float(BASELINE_XGB_METRICS['PR-AUC'])
    delta_f1 = tabm_metrics['F1@0.5'] - float(BASELINE_XGB_METRICS['F1'])
    fold_deltas = [float(row['delta_gini_vs_xgboost']) for row in folds]
    wins = sum(delta > 0.0 for delta in fold_deltas)
    precision_down = tabm_metrics['Precision@0.5'] < float(BASELINE_XGB_METRICS['Precision'])
    recall_down = tabm_metrics['Recall@0.5'] < float(BASELINE_XGB_METRICS['Recall'])
    if runtime_seconds > RUNTIME_BUDGET_SECONDS:
        return 'inferior_or_impractical'
    if (delta_gini >= 0.0100 and delta_pr_auc >= 0.0050 and wins >= 2
            and min(fold_deltas) >= -0.0030 and delta_f1 >= -0.010
            and not (precision_down and recall_down)):
        return 'material_gain'
    if delta_gini <= -0.0100 and delta_pr_auc <= -0.0050 and sum(delta < 0.0 for delta in fold_deltas) >= 2:
        return 'inferior_or_impractical'
    return 'comparable'


## Controlled run

Execute this cell only when the full controlled experiment is authorised. It reads the baseline metrics; it does not retrain GBDT and never evaluates the final test.

In [3]:
started_at = time.perf_counter()
run_status = 'completed'
fold_rows: list[dict[str, object]] = []
oof_probability: np.ndarray | None = None
oof_fold: np.ndarray | None = None

try:
    if sha256_file(DATASET) != EXPECTED_DATASET_SHA256:
        raise ValueError('Dataset SHA-256 не совпадает с зафиксированным contract.')
    data = pd.read_excel(DATASET, engine='pyxlsb')
    allowed_columns = [
        column for column in data.columns
        if column not in [TARGET, IDENTIFIER, *FORBIDDEN_FEATURES]
    ]
    if allowed_columns != FEATURES:
        raise ValueError('Feature identity или порядок отличается от Stage 1 reference.')
    if any(name in FEATURES or name in allowed_columns for name in FORBIDDEN_FEATURES):
        raise ValueError('Q_B1_norm и Q_B2_norm запрещены в FEATURES и X.')
    x_all = data.loc[:, FEATURES]
    y_all = data[TARGET].to_numpy(dtype=np.int64)
    if list(x_all.columns) != FEATURES or x_all.shape[1] != 47:
        raise ValueError('Нарушен точный порядок или число разрешённых признаков.')
    x_all = x_all.to_numpy(dtype=np.float32)
    if not np.isfinite(x_all).all():
        raise ValueError('Обнаружены NaN или non-finite значения; imputation запрещён.')

    all_indices = np.arange(len(data))
    working_indices, final_test_indices = train_test_split(
        all_indices, test_size=0.20, stratify=y_all, random_state=OUTER_SEED,
    )
    if len(working_indices) != 289614 or len(final_test_indices) != 72404:
        raise ValueError('Размер split не совпадает с experiment lock.')
    if hash_indices(working_indices) != EXPECTED_WORKING_INDEX_SHA256:
        raise ValueError('SHA рабочих индексов не совпадает с Stage 1 contract.')
    x_work, y_work = x_all[working_indices], y_all[working_indices]
    del data, x_all, y_all, final_test_indices  # final test не материализуется

    outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=OUTER_SEED)
    oof_probability = np.full(len(y_work), np.nan, dtype=np.float32)
    oof_fold = np.zeros(len(y_work), dtype=np.int8)
    for fold_number, (train_idx, valid_idx) in enumerate(outer_cv.split(x_work, y_work), start=1):
        fold_seed = FOLD_SEEDS[fold_number]
        best_epoch, inner_auc = select_best_epoch(x_work[train_idx], y_work[train_idx], fold_seed, started_at)
        valid_probability = refit_and_predict(
            x_work[train_idx], y_work[train_idx], x_work[valid_idx], fold_seed, best_epoch, started_at,
        )
        oof_probability[valid_idx] = valid_probability
        oof_fold[valid_idx] = fold_number
        tabm_fold_metrics = metrics(y_work[valid_idx], valid_probability)
        xgb_fold = BASELINE_XGB_FOLDS[fold_number - 1]
        fold_rows.append({
            'fold': fold_number, 'seed': fold_seed, 'best_epoch': best_epoch,
            'inner_best_roc_auc': inner_auc, 'tabm': tabm_fold_metrics,
            'xgboost_reference': xgb_fold,
            'delta_gini_vs_xgboost': tabm_fold_metrics['Gini'] - float(xgb_fold['Gini']),
            'delta_pr_auc_vs_xgboost': tabm_fold_metrics['PR-AUC'] - float(xgb_fold['PR-AUC']),
        })
    check_budget(started_at)

except RuntimeBudgetExceeded:
    run_status = 'runtime_budget_exceeded'

runtime_seconds = time.perf_counter() - started_at
if run_status == 'completed':
    try:
        check_budget(started_at)
    except RuntimeBudgetExceeded:
        run_status = 'runtime_budget_exceeded'
if run_status == 'completed':
    assert oof_probability is not None and np.isfinite(oof_probability).all()
    tabm_oof_metrics = metrics(y_work, oof_probability)
    decision_class = decide(tabm_oof_metrics, fold_rows, runtime_seconds)
else:
    tabm_oof_metrics = None
    decision_class = 'inferior_or_impractical'

metadata = {
    'experiment': 'Stage 6', 'version': 'V1', 'status': run_status,
    'question': 'TabM vs сохранённый Stage 1 GBDT baseline на 47 разрешённых признаках',
    'dataset_sha256': EXPECTED_DATASET_SHA256, 'target': TARGET, 'identifier': IDENTIFIER,
    'feature_names_in_order': FEATURES,
    'feature_identity_sha256': hashlib.sha256('\n'.join(FEATURES).encode()).hexdigest(),
    'working_rows': 289614, 'final_test_rows': 72404,
    'working_index_sha256': EXPECTED_WORKING_INDEX_SHA256, 'final_test_used': False,
    'outer_cv': {'type': 'StratifiedKFold', 'n_splits': 3, 'shuffle': True, 'random_state': 42},
    'fold_seeds': FOLD_SEEDS, 'tabm_config': TABM_CONFIG,
    'early_stopping': {'inner_train_fraction': 0.90, 'inner_validation_fraction': 0.10, 'patience': PATIENCE, 'min_delta': 0, 'selection_metric': 'ROC-AUC', 'refit_epochs': 'best_epoch + 1'},
    'baseline': {'primary': 'XGBoost', 'oof_metrics': BASELINE_XGB_METRICS, 'fold_metrics': BASELINE_XGB_FOLDS, 'context_only': {name: baseline['модели'][name]['итоговые_метрики'] for name in ['CatBoost', 'LightGBM']}},
    'fold_metrics': fold_rows, 'oof_metrics': tabm_oof_metrics, 'runtime_seconds': runtime_seconds,
    'runtime_budget_seconds': RUNTIME_BUDGET_SECONDS, 'decision_class': decision_class,
    'versions': {'python': platform.python_version(), 'os': platform.platform(), 'cpu_count': os.cpu_count(), 'torch': torch.__version__, 'tabm': tabm.__version__, 'numpy': np.__version__, 'torch_num_threads': torch.get_num_threads()},
    'limitations': ['3 folds не являются основанием для statistical significance claim.', 'Random CV не доказывает temporal stability.', 'Final test не использован.', 'Threshold 0.5 — только диагностический.'],
}
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
(GENERATED_DIR / 'stage6_tabm_results_V1.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
(SUMMARY_DIR / 'stage6_tabm_summary_V1.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
if run_status == 'completed':
    np.savez_compressed(GENERATED_DIR / 'stage6_tabm_oof_V1.npz', y_true=y_work, oof_probability=oof_probability, fold=oof_fold)
print({'status': run_status, 'decision_class': decision_class, 'runtime_seconds': runtime_seconds})


{'status': 'runtime_budget_exceeded', 'decision_class': 'inferior_or_impractical', 'runtime_seconds': 28800.3635327}


## Structured conclusion

After a completed run, use only the saved OOF metrics and the implemented rule: `material_gain`, `comparable`, or `inferior_or_impractical`. No claim of statistical significance is made from three folds. A runtime-budget stop is recorded as `runtime_budget_exceeded` and is not resumed.